# MLFlow Project Tracking

In [39]:
import os
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
client = mlflow.MlflowClient()

PROJECT_DIR = "iris_project"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project directory:", os.path.abspath(PROJECT_DIR))

Project directory: /home/pakshal/lab/q4/iris_project


## Data generation and storage

In [40]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

data_path = os.path.join(PROJECT_DIR, 'data')
os.makedirs(data_path, exist_ok=True)
X_train.to_csv(os.path.join(data_path, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(data_path, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(data_path, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(data_path, "y_test.csv"), index=False)

## Train script

In [41]:
%%writefile iris_project/train.py
import os
import argparse
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=3)
    parser.add_argument("--max_depth", type=int, default=3)
    parser.add_argument("--data_path", type=str, default="data")
    args = parser.parse_args()

    mlflow.set_tracking_uri("http://localhost:5000")

    X_train = pd.read_csv(os.path.join(args.data_path, "X_train.csv"))
    X_test = pd.read_csv(os.path.join(args.data_path, "X_test.csv"))
    y_train = pd.read_csv(os.path.join(args.data_path, "y_train.csv")).values.ravel()
    y_test = pd.read_csv(os.path.join(args.data_path, "y_test.csv")).values.ravel()

    with mlflow.start_run(run_name=f"project-run-n{args.n_estimators}-d{args.max_depth}"):
        mlflow.log_param("n_estimators", args.n_estimators)
        mlflow.log_param("max_depth", args.max_depth)

        model = RandomForestClassifier(
            n_estimators=args.n_estimators, max_depth=args.max_depth, random_state=42
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)
        mlflow.sklearn.log_model(model, name="model")

        print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}  run_id={mlflow.active_run().info.run_id}")


if __name__ == "__main__":
    main()

Writing iris_project/train.py


## YAML Files

In [42]:
%%writefile iris_project/MLproject
name: iris-classifier

python_env: python_env.yaml

entry_points:
  main:
    parameters:
      n_estimators: {type: int, default: 3}
      max_depth: {type: int, default: 3}
      data_path: {type: str, default: "data"}
    command: "python train.py --n_estimators {n_estimators} --max_depth {max_depth} --data_path {data_path}"

Writing iris_project/MLproject


In [43]:
%%writefile iris_project/python_env.yaml
python: "3.12"
build_dependencies:
  - pip
  - setuptools
dependencies:
  - scikit-learn
  - pandas
  - mlflow

Writing iris_project/python_env.yaml


## Main run

In [44]:
!mlflow run iris_project -P n_estimators=3 -P max_depth=3 \
    --experiment-name iris-classifier --env-manager=local

2026/08/30 21:49:05 INFO mlflow.projects: 'iris-classifier' does not exist. Creating a new experiment
2026/08/30 21:49:05 INFO mlflow.projects.utils: === Created directory /tmp/tmpyfdt692k for downloading remote URIs passed to arguments of type 'path' ===
2026/08/30 21:49:05 INFO mlflow.projects.backend.local: === Running command 'python train.py --n_estimators 3 --max_depth 3 --data_path data' in run with ID '1787848fd96847a99bdc2d38c8994d9e' === 
Uploading artifacts: 100%|███████████████████████| 5/5 [00:03<00:00,  1.56it/s]
accuracy=1.0000  f1_macro=1.0000  run_id=1787848fd96847a99bdc2d38c8994d9e
🏃 View run project-run-n3-d3 at: http://localhost:5000/#/experiments/1/runs/1787848fd96847a99bdc2d38c8994d9e
🧪 View experiment at: http://localhost:5000/#/experiments/1
2026/08/30 21:49:20 INFO mlflow.projects: === Run (ID '1787848fd96847a99bdc2d38c8994d9e') succeeded ===
🏃 View run project-run-n3-d3 at: http://localhost:5000/#/experiments/1/runs/1787848fd96847a99bdc2d38c8994d9e
🧪 View expe

## Register the model and transition to Staging

In [45]:
import mlflow
runs_df = mlflow.search_runs(
    experiment_names=["iris-classifier"],
    order_by=["start_time DESC"],
)
run_id = runs_df['run_id'][0]
print(run_id)

1787848fd96847a99bdc2d38c8994d9e


In [46]:
result = mlflow.register_model(model_uri=f"runs:/{run_id}/model", name="iris-rf")
client.transition_model_version_stage(name=result.name, version=result.version, stage="Staging")

Successfully registered model 'iris-rf'.
2026/08/30 21:49:36 WARNING mlflow.tracking._model_registry.fluent: Run with id 1787848fd96847a99bdc2d38c8994d9e has no artifacts at artifact path 'model', registering model based on models:/m-054a75765f7d4e47bd805dcfb511f530 instead
2026/08/30 21:49:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-rf, version 1
Created version '1' of model 'iris-rf'.
/tmp/ipykernel_69838/56011707.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(name=result.name, version=result.version, stage="Staging")


<ModelVersion: aliases=[], creation_timestamp=1788106776693, current_stage='Staging', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1788106777045, metrics=None, model_id=None, name='iris-rf', params=None, run_id='1787848fd96847a99bdc2d38c8994d9e', run_link='', source='models:/m-054a75765f7d4e47bd805dcfb511f530', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>